In [1]:
import warnings
from numba.core.errors import NumbaWarning

warnings.simplefilter('ignore', category=NumbaWarning)

In [2]:
import IPython.display as ipd
import torch
from torch.utils.data import DataLoader

import commons
import utils
from data_utils import TextAudioSpeakerLoader, TextAudioSpeakerCollate
from models import SynthesizerTrn
from text.symbols import symbols
from text import text_to_sequence


def get_text(text, hps):
    text_norm = text_to_sequence(text, hps.data.text_cleaners)
    if hps.data.add_blank:
        text_norm = commons.intersperse(text_norm, 0)
    text_norm = torch.LongTensor(text_norm)
    return text_norm

DEBUG:numba.core.byteflow:bytecode dump:
>          0	NOP(arg=None, lineno=1023)
           2	RESUME(arg=0, lineno=1023)
           4	LOAD_FAST(arg=0, lineno=1026)
           6	LOAD_CONST(arg=1, lineno=1026)
           8	BINARY_SUBSCR(arg=None, lineno=1026)
          12	LOAD_FAST(arg=0, lineno=1026)
          14	LOAD_CONST(arg=2, lineno=1026)
          16	BINARY_SUBSCR(arg=None, lineno=1026)
          20	COMPARE_OP(arg=68, lineno=1026)
          24	LOAD_FAST(arg=0, lineno=1026)
          26	LOAD_CONST(arg=1, lineno=1026)
          28	BINARY_SUBSCR(arg=None, lineno=1026)
          32	LOAD_FAST(arg=0, lineno=1026)
          34	LOAD_CONST(arg=3, lineno=1026)
          36	BINARY_SUBSCR(arg=None, lineno=1026)
          40	COMPARE_OP(arg=92, lineno=1026)
          44	BINARY_OP(arg=1, lineno=1026)
          48	RETURN_VALUE(arg=None, lineno=1026)
DEBUG:numba.core.byteflow:pending: deque([State(pc_initial=0 nstack_initial=0)])
DEBUG:numba.core.byteflow:stack: []
DEBUG:numba.core.byteflow:state.

## LJ Speech

In [ ]:
hps = utils.get_hparams_from_file("./configs/ljs_base.json")

In [ ]:
net_g = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    **hps.model).cuda()
_ = net_g.eval()

_ = utils.load_checkpoint("./G_141000.pth", net_g, None)

/home/yash7/.local/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


INFO:root:Loaded checkpoint './G_141000.pth' (iteration 180)


In [ ]:
texts = ["This is a generated sentence", "These were not present in the dataset", "Some words such as LJSpeech need pronunciations for individual letters"]

for text in texts:
    stn_tst = get_text(text, hps)
    with torch.no_grad():
        x_tst = stn_tst.cuda().unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
        audio = net_g.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
    ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=False))

## wTIMIT

In [3]:
hps = utils.get_hparams_from_file("./configs/wtimit_base.json")
# hps.model.gin_channels = 96

In [4]:
net_g = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    n_speakers=hps.data.n_speakers,
    **hps.model).cuda()
_ = net_g.eval()

_ = utils.load_checkpoint("G_225000.pth", net_g, None)

/home/yash7/.local/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


INFO:root:Loaded checkpoint 'G_225000.pth' (iteration 87)


In [6]:
stn_tst = get_text("VITS has a voice-conversion pipeline in the provided code. Maybe it can be used for directly converting from whisper to speech", hps)
with torch.no_grad():
    x_tst = stn_tst.cuda().unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
    print(x_tst.shape, x_tst_lengths)
    sid = torch.LongTensor([3]).cuda()
    audio = net_g.infer(x_tst, x_tst_lengths, sid=sid, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=True))
# audio = net_g(x_tst, x_tst_lengths, sid=sid, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()

with torch.no_grad():
    x_tst = stn_tst.cuda().unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
    print(x_tst.shape, x_tst_lengths)
    sid = torch.LongTensor([2]).cuda()
    audio = net_g.infer(x_tst, x_tst_lengths, sid=sid, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=True))

torch.Size([1, 271]) tensor([271], device='cuda:0')
torch.Size([1, 192, 680])


torch.Size([1, 271]) tensor([271], device='cuda:0')
torch.Size([1, 192, 600])


### Voice Conversion

In [6]:
lines = []
with open('./filelists/wtimit_val.txt.cleaned', 'r+') as readFile:
    lines = readFile.readlines()
lines = [l.replace('/ssd_scratch/cvit/yash/converted_wavs/normal/', '/home/yash7/vF/valFiles/') for l in lines]
lines = [l.replace('/wavs/', '/') for l in lines]
with open('./filelists/wtimit_val2.txt.cleaned', 'w') as readFile:
    readFile.writelines(''.join(lines))

In [7]:
dataset = TextAudioSpeakerLoader("./test.txt.cleaned", hps.data)
collate_fn = TextAudioSpeakerCollate()
loader = DataLoader(dataset, num_workers=8, shuffle=False,
    batch_size=1, pin_memory=True,
    drop_last=True, collate_fn=collate_fn)
data_list = list(loader)

DEBUG:numba.core.byteflow:bytecode dump:
>          0	NOP(arg=None, lineno=1137)
           2	RESUME(arg=0, lineno=1137)
           4	LOAD_FAST(arg=0, lineno=1140)
           6	LOAD_CONST(arg=1, lineno=1140)
           8	BINARY_SUBSCR(arg=None, lineno=1140)
          12	STORE_FAST(arg=3, lineno=1140)
          14	LOAD_FAST(arg=1, lineno=1141)
          16	UNARY_NEGATIVE(arg=None, lineno=1141)
          18	LOAD_FAST(arg=3, lineno=1141)
          20	SWAP(arg=2, lineno=1141)
          22	COPY(arg=2, lineno=1141)
          24	COMPARE_OP(arg=26, lineno=1141)
          28	POP_JUMP_IF_FALSE(arg=5, lineno=1141)
          30	LOAD_FAST(arg=1, lineno=1141)
          32	COMPARE_OP(arg=26, lineno=1141)
          36	POP_JUMP_IF_FALSE(arg=5, lineno=1141)
          38	JUMP_FORWARD(arg=2, lineno=1141)
>         40	POP_TOP(arg=None, lineno=1141)
          42	JUMP_FORWARD(arg=2, lineno=1141)
>         44	LOAD_CONST(arg=1, lineno=1142)
          46	STORE_FAST(arg=3, lineno=1142)
>         48	LOAD_FAST(arg

In [10]:
with torch.no_grad():
    x, x_lengths, spec, spec_lengths, y, y_lengths, sid_src = [x.cuda() for x in data_list[0]]
    print(spec.shape, spec_lengths)
    sid_tgt1 = torch.LongTensor([38]).cuda()
    sid_tgt2 = torch.LongTensor([53]).cuda()
    sid_tgt3 = torch.LongTensor([54]).cuda()
    audio1 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt1)[0][0,0].data.cpu().float().numpy()
    audio2 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt2)[0][0,0].data.cpu().float().numpy()
    audio3 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt3)[0][0,0].data.cpu().float().numpy()
    audio4 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_src)[0][0,0].data.cpu().float().numpy()
    print(y.shape, audio1.shape, audio2.shape, audio3.shape)    
# print(len(audio1), len(audio1[0]), len(audio1[0][0]))
print("Original SID: %d" % sid_src.item())
ipd.display(ipd.Audio(y[0].cpu().numpy(), rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_tgt1.item())
ipd.display(ipd.Audio(audio1, rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_tgt2.item())
ipd.display(ipd.Audio(audio2, rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_tgt3.item())
ipd.display(ipd.Audio(audio3, rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_src.item())
ipd.display(ipd.Audio(audio4, rate=hps.data.sampling_rate, normalize=True))

torch.Size([1, 513, 441]) tensor([441], device='cuda:0')
PostEnc Inp:  torch.Size([1, 513, 441]) tensor([441], device='cuda:0')
torch.Size([1, 192, 441]) torch.Size([1, 1, 112896])
PostEnc Inp:  torch.Size([1, 513, 441]) tensor([441], device='cuda:0')
torch.Size([1, 192, 441]) torch.Size([1, 1, 112896])
PostEnc Inp:  torch.Size([1, 513, 441]) tensor([441], device='cuda:0')
torch.Size([1, 192, 441]) torch.Size([1, 1, 112896])
PostEnc Inp:  torch.Size([1, 513, 441]) tensor([441], device='cuda:0')
torch.Size([1, 192, 441]) torch.Size([1, 1, 112896])
torch.Size([1, 1, 113087]) (112896,) (112896,) (112896,)
Original SID: 55


Converted SID: 38


Converted SID: 53


Converted SID: 54


Converted SID: 55


In [ ]:
from mel_processing import spectrogram_torch
import os

def get_audio(filename):
    audio, sampling_rate = utils.load_wav_to_torch(filename)
    audio_norm = audio / 1.13
    audio_norm = audio_norm.unsqueeze(0)
    spec = spectrogram_torch(audio_norm, 1024,
        24000, 256, 1024,
        center=False)
    spec = torch.squeeze(spec, 0)
    return spec, audio_norm

In [7]:
from models import SynthesizerTrnCopy
import nemo.collections.asr as nemo_asr

netG = SynthesizerTrnCopy(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    n_speakers=hps.data.n_speakers,
    **hps.model
).cuda()
_ = netG.eval()

_ = utils.load_checkpoint("G_174000.pth", netG, None)
speakerModel = nemo_asr.models.EncDecSpeakerLabelModel.from_pretrained("nvidia/speakerverification_en_titanet_large")
speakerModel.eval()
speakerModel = speakerModel.cuda()
speakerEmbed = speakerModel.get_embedding('/home/yash7/LibriSpeech/dev-clean/174/84280/174-84280-0000.flac').transpose(-2, -1).unsqueeze(0)
print(speakerEmbed.shape)
stn_tst = get_text("VITS has a voice-conversion pipeline in the provided code. Maybe it can be used for directly converting from whisper to speech", hps)
with torch.no_grad():
    x_tst = stn_tst.cuda().unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
    audio = netG.infer(x_tst, x_tst_lengths, g = speakerEmbed, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=True))

INFO:root:Loaded checkpoint 'G_174000.pth' (iteration 84)
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): huggingface.co:443
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /nvidia/speakerverification_en_titanet_large/resolve/main/speakerverification_en_titanet_large.nemo HTTP/1.1" 302 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /nvidia/speakerverification_en_titanet_large/resolve/main/speakerverification_en_titanet_large.nemo HTTP/1.1" 302 0


[NeMo W 2025-04-21 05:58:45 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
        min_speed_rate: 0.95
        max_speed_rate: 1.05
    num_workers: 15
    pin_memory: true
    
[NeMo W 2025-04-21 05:58:45 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data

[NeMo I 2025-04-21 05:58:45 nemo_logging:393] PADDING: 16
[NeMo I 2025-04-21 05:58:46 nemo_logging:393] Model EncDecSpeakerLabelModel was successfully restored from /home/yash7/.cache/huggingface/hub/models--nvidia--speakerverification_en_titanet_large/snapshots/0dc382f40121a5fbd34db10a2bb04d826c2be6a8/speakerverification_en_titanet_large.nemo.
torch.Size([1, 192, 1])


In [10]:
speakerModel = nemo_asr.models.EncDecSpeakerLabelModel.from_pretrained("nvidia/speakerverification_en_titanet_large")
speakerModel.eval()
speakerModel = speakerModel.cuda()
speakerEmbed = speakerModel.get_embedding('/home/yash7/LibriSpeech/dev-clean/8842/302201/8842-302201-0000.flac').transpose(-2, -1).unsqueeze(0)
with torch.no_grad():
    x, x_lengths, spec, spec_lengths, y, y_lengths, sid_src = [x.cuda() for x in data_list[0]]
    audio1 = netG.voice_conversion(spec, spec_lengths, sid_src=sid_src, g_tgt=speakerEmbed)[0][0,0].data.cpu().float().numpy()
# print(len(audio1), len(audio1[0]), len(audio1[0][0]))
print(len(audio1), len(x[0]))
print("Original SID: %d" % sid_src.item())
ipd.display(ipd.Audio(y[0].cpu().numpy(), rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: Test" )
ipd.display(ipd.Audio(audio1, rate=hps.data.sampling_rate, normalize=True))

DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /nvidia/speakerverification_en_titanet_large/resolve/main/speakerverification_en_titanet_large.nemo HTTP/1.1" 302 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /nvidia/speakerverification_en_titanet_large/resolve/main/speakerverification_en_titanet_large.nemo HTTP/1.1" 302 0


[NeMo W 2025-04-21 06:01:59 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
        min_speed_rate: 0.95
        max_speed_rate: 1.05
    num_workers: 15
    pin_memory: true
    
[NeMo W 2025-04-21 06:01:59 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data

[NeMo I 2025-04-21 06:01:59 nemo_logging:393] PADDING: 16
[NeMo I 2025-04-21 06:01:59 nemo_logging:393] Model EncDecSpeakerLabelModel was successfully restored from /home/yash7/.cache/huggingface/hub/models--nvidia--speakerverification_en_titanet_large/snapshots/0dc382f40121a5fbd34db10a2bb04d826c2be6a8/speakerverification_en_titanet_large.nemo.
112896 105
Original SID: 55


Converted SID: Test


In [ ]:
## TODO: In progress
audioWavPath = ''
spec, wav = get_audio(audioWavPath)

with torch.no_grad():
    x, x_lengths, spec, spec_lengths, y, y_lengths, sid_src = [x.cuda() for x in data_list[0]]
    sid_tgt1 = torch.LongTensor([38]).cuda()
    sid_tgt2 = torch.LongTensor([53]).cuda()
    sid_tgt3 = torch.LongTensor([4]).cuda()
    audio1 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt1)[0][0,0].data.cpu().float().numpy()
    audio2 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt2)[0][0,0].data.cpu().float().numpy()
    audio3 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt3)[0][0,0].data.cpu().float().numpy()
    audio4 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_src)[0][0,0].data.cpu().float().numpy()
# print(len(audio1), len(audio1[0]), len(audio1[0][0]))
print(len(audio1), len(x[0]))
print("Original SID: %d" % sid_src.item())
ipd.display(ipd.Audio(y[0].cpu().numpy(), rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_tgt1.item())
ipd.display(ipd.Audio(audio1, rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_tgt2.item())
ipd.display(ipd.Audio(audio2, rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_tgt3.item())
ipd.display(ipd.Audio(audio3, rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_src.item())
ipd.display(ipd.Audio(audio4, rate=hps.data.sampling_rate, normalize=True))